In [0]:
import requests

In [0]:
import requests
all_data = []
limit = 50
skip = 0
while True:
    url = f"https://dummyjson.com/users?limit={limit}&skip={skip}"
    response = requests.get(url).json()
    users = response.get("users", [])
    if not users:
        break
    all_data.extend(users)
    skip += limit
print(f"Total records fetched: {len(all_data)}")


In [0]:
from pyspark.sql.types import *
user_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("address", StructType([
        StructField("address", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True)]))
])

df = spark.createDataFrame(data, schema=user_schema)
display(df)


In [0]:
from pyspark.sql.functions import *
df = df.withColumn("address_array", array("address"))
df = df.withColumn("address_explode", explode("address_array"))
df_flat = df.select(
    "id",
    "firstName",
    "lastName",
    "email",
    "phone",
    "age",
    "gender",
    col("address_explode.address").alias("street"),
    col("address_explode.city").alias("city"),
    col("address_explode.state").alias("state")
)
display(df_flat)

In [0]:
from pyspark.sql.functions import *
df_flat = df_flat.withColumn("site_address", split("email", "@")[1])
display(df_flat)

In [0]:
from pyspark.sql.functions import *
df_flat = df_flat.withColumn("load_date", current_date())
display(df_flat)

In [0]:
df_flat.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("/Volumes/question_2/silver/site_info")